In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

### Задание 1
Напишите функцию ```get_normalize```, которая будет принимать тензор с признаками объектов из какого-то датасета с картинками и будет возвращать поканальное среднее и поканальное стандартное отклонение. Гарантируется, что матрица будет иметь размер ```[N, C, H, W]```, где ```N``` это количество объектов, ```C``` — количество каналов, ```H, W``` — размеры изображений. Нужно вернуть кортеж из двух тензоров длины C.

Ваша функция должна иметь следующую сигнатуру ```def get_normalize(features: torch.Tensor):```



In [2]:
def get_normalize(features: torch.Tensor):
    mean = features.mean(dim=(0, 2, 3))
    std = features.std(dim=(0, 2, 3))
    return mean, std

### Задание 2
Напишите функцию get_augmentations, которая будет возвращать готовые аугментации для обучающей выборки и для тестовой выборки. Она должна иметь следующую сигнатуру: def get_augmentations(train: bool = True) -> T.Compose:.

Примените следующие аугментации:

 Измените размер изображения, чтобы его размер был 224 на 224 пикселя (и для обучающей и для тестовой выборки).
 Примените какие-нибудь аугментации из тех, что мы изучили на занятии (только для обучающей).
 Преобразуйте картинку в тензор.
 Примените нормализацию для датасета CIFAR10
Понятно, что изменение размера к 224 на 224 и нормализация для CIFAR10 вместе сочетаются плохо. Поэтому в дальнейшем, когда будете использовать эту функцию для получения аугментаций для вашего нового датасета, не забудьте поменять их на специфичные ему.

In [3]:
train_dataset = CIFAR10('../datasets/cifar10',train=True, download = True)   
test_dataset = CIFAR10('../datasets/cifar10',train=False, download=True)

Files already downloaded and verified
Files already downloaded and verified


In [4]:
print(f'размерность датасета cifar10 = {train_dataset.data.shape}')

размерность датасета cifar10 = (50000, 32, 32, 3)


In [5]:
def get_normalize(features):
    means = (features.data/255).mean(axis=(0, 1, 2))
    stds = (features.data/255).std(axis=(0, 1, 2))
    return means, stds

In [6]:
means, stds = get_normalize(train_dataset)
means, stds

(array([0.49139968, 0.48215841, 0.44653091]),
 array([0.24703223, 0.24348513, 0.26158784]))

In [7]:
def get_augmentations(train: bool=True) -> T.Compose:
    if train:
        train_transforms = T.Compose(
            [T.Resize((224,224)),
             T.RandomHorizontalFlip(p=0.7),
             T.RandomInvert(p=0.5),
             T.ToTensor(),
             T.Normalize(mean = means, std = stds)
            ]
        )
        return train_transforms
    else:
        test_transforms = T.Compose(
            [T.Resize((224,224)),
             T.ToTensor(),
             T.Normalize(mean = means, std = stds)
            ]
        )
        return test_transforms

In [8]:
train_transforms = get_augmentations(train = True)
train_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.7)
    RandomInvert(p=0.5)
    ToTensor()
    Normalize(mean=[0.49139968 0.48215841 0.44653091], std=[0.24703223 0.24348513 0.26158784])
)

In [9]:
test_transforms = get_augmentations(train = False)
test_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.49139968 0.48215841 0.44653091], std=[0.24703223 0.24348513 0.26158784])
)

### Задание 3
Напишите функцию predict. Она должна принимать на вход нейронную сеть, даталоадер и torch.device. Она должна иметь следующую сигнатуру: def predict(model: nn.Module, loader: DataLoader, device: torch.device):

Внутри функции сделайте следующие шаги:

Создайте пустой список для хранения предсказаний.
Проитерируйтесь по даталоадеру.
На каждой итерации сделайте forward pass для батча, посчитайте классы как аргмакс по выходу нейросети, по логитам, добавьте тензор с предсказаниями в список.
Сделайте конкатенацию всех предсказаний и верните этот тензор длины N, по числу объектов в датасете.
Ваша функция должна возвращать тензор с классами.

In [10]:
@torch.inference_mode()  
# Отключает вычисление градиентов.
# Не строится граф вычислений → меньше памяти и быстрее инференс.

def predict(model: nn.Module, loader: DataLoader, device: torch.device):
    
    model.eval()
    # Перевод модели в режим инференса:
    # - Dropout отключается
    # - BatchNorm использует накопленные статистики
    
    preds = []
    # Список для хранения предсказаний по батчам.
    # Мы будем добавлять сюда тензоры размера [B].

    for x, y in tqdm(loader):
        # loader возвращает батчи:
        # x → изображения формы [B, C, H, W]
        # y → истинные метки формы [B]

        x, y = x.to(device), y.to(device)
        # Перенос данных на нужное устройство (CPU или GPU).
        # Без этого модель и данные могут оказаться на разных устройствах → ошибка.

        logits = model(x)
        # Forward pass.
        # logits имеет форму [B, C], где:
        # B — размер батча
        # C — число классов
        # Это сырые выходы (логиты), не вероятности.

        pred = torch.argmax(logits, dim=1)
        # argmax выбирает индекс максимального значения.
        #
        # logits: [B, C]
        # dim=1 означает:
        #   искать максимум вдоль размерности классов.
        #
        # То есть для каждой строки (каждого объекта)
        # выбирается класс с максимальным логитом.
        #
        # Результат: тензор формы [B]
        # Один индекс класса на один объект.

        preds.append(pred.cpu())
        # Перенос предсказаний на CPU (если модель была на GPU).
        # Добавляем батч предсказаний в список.

    preds_tensor = torch.cat(preds)
    # torch.cat склеивает список тензоров в один.
    #
    # preds — это список тензоров формы [B].
    #
    # cat по умолчанию делает concat по dim=0,
    # то есть "в длину".
    #
    # Если всего N объектов в датасете,
    # результат будет формы [N].

    return preds_tensor
    # Возвращаем итоговый тензор предсказаний длины N.

### Задание 5
Напишите функцию ```predict_tta```. Она должна принимать на вход нейронную сеть, даталоадер, torch.device и количество итераций по даталоадеру. Она должна иметь следующую сигнатуру: ```def predict_tta(model: nn.Module, loader: DataLoader, device: torch.device, iterations: int = 2):```.

В этой функции мы применим технику, которую кратко обсуждали на занятии  — Test Time Augmentation. Основная идея заключается в том, чтобы сделать для каждой картинки из тестовой выборки несколько аугментированных вариантов и сделать для них предсказания. Потом эти предсказания усреднить и использовать как обычно. В синтаксисе PyTorch это вырождается в создание тестового датасета со случайными аугментациями (либо как на обучающей выборке, либо чуть более слабыми). После этого нужно проитерироваться по созданному датасету несколько раз и усреднить ответы модели.

Внутри функции сделайте следующие шаги:

Запустите цикл по количеству итераций.
Внутри цикла проитерируйтесь по даталоадеру.
Запишите ответы (не классы, а сырые выходы нейросети) модели в один большой тензор размера ```[N, C]```, где ```C``` — число классов, а ```N``` — количество объектов в датасете (то есть мы должны иметь для каждого объекта вектор из выходов нейросети, логиты).
Сделайте из этих тензоров один огромный тензор размера ```[N, C, iterations]```, усредните его по итерациям, чтобы его размер стал ```[N, C]```
Дальше предскажите классы для объектов по этому тензору как аргмакс, верните их из функции.
Ваша функция должна возвращать тензор с классами. Не забудьте переводить модель в режим применения и навешивать декторатор для выключения подсчета градиентов.

In [11]:
@torch.inference_mode()
def predict_tta(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    iterations: int = 2
):
    model.eval()

    tta_logits = []  # сюда будем складывать [N, C], где N — количество объектов, C — количество классов для каждой итерации, два таких тензора соответсвенно по количеству операций

    for _ in range(iterations):

        all_logits = []  # логиты за проход по датасету, будет столько тензорв размерности [B, C] в списке all_logits, сколько всего батчей

        for x, y in tqdm(loader):
            x = x.to(device)

            logits = model(x)  # [B, C]
            all_logits.append(logits.cpu())  # переносим на CPU, чтобы не переполнить GPU

        all_logits = torch.cat(all_logits, dim=0)  # объединяем тензоры по строкам так, что B+B+B+.. = N --> [N, C] итоговая размерность
        tta_logits.append(all_logits) # аналогично складываем в нашем случае два тензора размером [N, C]

    # делаем [N, C, iterations]
    tta_logits = torch.stack(tta_logits, dim=2) # создаем третью ось, как будто в 3 д, у нас будет по оси икс количество объектов (N штук), 
    # по y классы (10 штук) , а по z теперь еще добавится один слой
    # грубо говоря для понимания было два 2D слоя (тензора) друг под другом (сверху и снизку),
    # а стало два 2D слоя друг за другом (спереди и сзазди), как в 3 д матрице

    # усредняем по итерациям
    mean_logits = tta_logits.mean(dim=2)  # [N, C] # усредняем по этим двум слоям( dim это не слой), для соответсвующих элементов

    # предсказываем классы
    preds = mean_logits.argmax(dim=1)  # [N], для каждого объекта оставили только самый вероятнейший класс, поэтому стал размер N по количеству объектов, вектор столбец


    return preds